# 01 — Synthetic Data Audit

Generate, prepare, and inspect the synthetic FI-2010-shaped limit-order-book fixture used by notebooks 01–05. This dataset is designed to validate the software pipeline.

In [2]:
from pathlib import Path
import subprocess
import sys

import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "src").exists():
    raise FileNotFoundError("Run this notebook from the repository or notebooks directory.")

sys.path.insert(0, str(REPO_ROOT / "src"))
RAW = REPO_ROOT / "data/sample/synthetic_fi2010.txt"
PROCESSED = REPO_ROOT / "data/processed/synthetic_h50"

print("Repository:", REPO_ROOT)
print("Raw synthetic data:", RAW)
print("Processed data:", PROCESSED)


Repository: /home/shuaigou2025/projects/cost-aware-limit-order-book-forecasting
Raw synthetic data: /home/shuaigou2025/projects/cost-aware-limit-order-book-forecasting/data/sample/synthetic_fi2010.txt
Processed data: /home/shuaigou2025/projects/cost-aware-limit-order-book-forecasting/data/processed/synthetic_h50


## Generate and prepare the fixture
The following two cells recreate the raw synthetic file and the normalized horizon-50 processed dataset. Skip them when those files already exist.

In [9]:
# python scripts/generate_synthetic.py --output data/sample/synthetic_fi2010.txt
subprocess.run(
    [
        sys.executable,
        "scripts/generate_synthetic.py",
        "--output",
        str(RAW.relative_to(REPO_ROOT)),
    ],
    cwd=REPO_ROOT,
    check=True,
)

Wrote synthetic software-test data to data/sample/synthetic_fi2010.txt


CompletedProcess(args=['/home/shuaigou2025/projects/cost-aware-limit-order-book-forecasting/.venv/bin/python', 'scripts/generate_synthetic.py', '--output', 'data/sample/synthetic_fi2010.txt'], returncode=0)

In [10]:
# python scripts/prepare_data.py --input data/sample/synthetic_fi2010.txt --output-dir data/processed/synthetic_h50 --horizon 50 --sequence-length 100 --normalize train --executable-prices
subprocess.run(
    [
        sys.executable,
        "scripts/prepare_data.py",
        "--input",
        str(RAW.relative_to(REPO_ROOT)),
        "--output-dir",
        str(PROCESSED.relative_to(REPO_ROOT)),
        "--horizon",
        "50",
        "--sequence-length",
        "100",
        "--normalize",
        "train",
        "--executable-prices",
    ],
    cwd=REPO_ROOT,
    check=True,
)

{
  "source": "data/sample/synthetic_fi2010.txt",
  "n_observations": 8000,
  "n_features": 40,
  "horizon": 50,
  "sequence_length": 100,
  "normalization": "train",
  "executable_prices": true,
  "train_boundary": 5600,
  "validation_boundary": 6800,
  "purge_size": 99,
  "split_sizes": {
    "train": 5451,
    "validation": 1051,
    "test": 1051
  },
  "label_mapping": {
    "0": "down",
    "1": "stationary",
    "2": "up"
  },
  "price_columns": {
    "best_ask": 0,
    "best_bid": 2
  },
  "warning": "Executable-price analysis enabled."
}


CompletedProcess(args=['/home/shuaigou2025/projects/cost-aware-limit-order-book-forecasting/.venv/bin/python', 'scripts/prepare_data.py', '--input', 'data/sample/synthetic_fi2010.txt', '--output-dir', 'data/processed/synthetic_h50', '--horizon', '50', '--sequence-length', '100', '--normalize', 'train', '--executable-prices'], returncode=0)

## Load the prepared dataset

Inspect the saved configuration before reviewing features and labels.

In [11]:
from lob_project.data.processed import load_processed

data = load_processed(PROCESSED)
pd.Series(data.metadata, name="Value").to_frame()

,Value
source,data/sample/synthetic_fi2010.txt
n_observations,8000
n_features,40
horizon,50
sequence_length,100
normalization,train
executable_prices,True
train_boundary,5600
validation_boundary,6800
purge_size,99


## Feature integrity

The processed input contains 40 normalized LOB features: price and volume at ten ask/bid levels.

In [12]:
features = np.asarray(data.features)
feature_summary = pd.DataFrame(
    {
        "Mean": features.mean(axis=0),
        "Std. dev.": features.std(axis=0),
        "Minimum": features.min(axis=0),
        "Maximum": features.max(axis=0),
    },
)
feature_summary.index = [f"feature_{index:02d}" for index in range(features.shape[1])]
feature_summary.index.name = "Feature"
feature_summary.head(10).style.format("{:.4f}")

,Mean,Std. dev.,Minimum,Maximum
Feature,,,,
feature_00,0.0660,0.8945,-1.5465,2.0816
feature_01,-0.0317,0.9777,-2.2180,7.1071
feature_02,0.0660,0.8945,-1.5498,2.0784
feature_03,0.0231,1.0077,-2.1461,5.8102
feature_04,0.0660,0.8945,-1.5465,2.0816
feature_05,-0.0329,0.9899,-2.3330,6.5280
feature_06,0.0660,0.8945,-1.5498,2.0783
feature_07,0.0226,1.0252,-2.3598,6.3944
feature_08,0.0660,0.8945,-1.5465,2.0816


In [13]:
labels = np.asarray(data.labels)
split_indices = (data.train_indices, data.validation_indices, data.test_indices)

assert features.shape == (len(labels), 40)
assert np.isfinite(features).all()
assert set(np.unique(labels)).issubset({0, 1, 2})
assert all(
    len(indices) > 0
    and indices.min() >= 0
    and indices.max() < len(labels)
    and np.all(np.diff(indices) > 0)
    for indices in split_indices
)
print(
    f"Core checks passed: {len(features):,} observations, "
    f"40 features, {len(np.unique(labels))} observed classes."
)

Core checks passed: 8,000 observations, 40 features, 3 observed classes.


## Training-label distribution

Classes are encoded as `0 = down`, `1 = stationary`, and `2 = up`. For this synthetic horizon-50 dataset, direction is based on the average future midprice—not executable trade profitability.

In [14]:
class_names = {0: "down", 1: "stationary", 2: "up"}
training_labels = labels[data.train_indices]
class_counts = (
    pd.Series(training_labels)
    .value_counts()
    .reindex(class_names, fill_value=0)
)
label_summary = pd.DataFrame(
    {
        "Class": [class_names[label] for label in class_counts.index],
        "Observations": class_counts.to_numpy(),
        "Share": class_counts.to_numpy() / class_counts.sum(),
    },
).set_index("Class")
label_summary.style.format({"Observations": "{:,}", "Share": "{:.2%}"})

,Observations,Share
Class,,
down,"2,910",53.38%
stationary,193,3.54%
up,"2,348",43.07%
